# Landscape analysis

In [ ]:
import matplotlib as mpl
import numpy as np
import seaborn as sns
import swisslandstats as sls

import pylandstats as pls

The land use/land cover (LULC) data used in this notebook ships with the docs in the `data` directory (see [A03-swisslandstats-preprocessing.ipynb](https://github.com/martibosch/pylandstats-notebooks/blob/main/notebooks/A03-swisslandstats-preprocessing.ipynb) for how it is derived from the raw SLS data).

We can load our landscape from a GeoTiff file:

In [ ]:
URBAN_CLASS_VAL = 1
AGRICULTURAL_CLASS_VAL = 2
input_filepath = "data/veveyse/LU18_4.tif"

In [ ]:
ls = pls.Landscape(input_filepath)

The Swiss Land Statistics (SLS) inventory distinguishes 27 land use/land cover classes, however, to simplify, this repository uses the classification aggregated into four main categories, i.e., *urban* (1), *agricultural* (3) *wooded areas* and (4) *unproductive areas*:

In [ ]:
ls.plot_landscape(cmap=sls.noas04_4_cmap, norm=sls.noas04_4_norm, legend=True)

## Computing metrics

The metrics can be computed at the patch, class and landscape level (see [the list of implemented metrics](https://pylandstats.readthedocs.io/en/latest/landscape.html#list-of-implemented-metrics))

### Patch-level metrics

The metrics can be computed at the patch level, that is, for each patch of the landscape:

In [ ]:
patch_metrics_df = ls.compute_patch_metrics_df()
patch_metrics_df.head()

We can operate upon `patch_metrics_df` as with any other pandas DataFrame. In this case, there are 206 patches, of which 193 are *urban* and 13 *non-urban*, as noted respectively by the values of 1 and 2 within the `class_val` column:

In [ ]:
patch_metrics_df["class_val"].value_counts()

We might also use methods from other libraries, such as matplotlib or numpy. For instance, in order to explore the size distribution of patches, we can also plot the distribution of the logarithm of `area` for *urban* and *agricultural* classes as follows:

In [ ]:
# for better-looking histograms
sns.set_theme()

ax = (
    patch_metrics_df[patch_metrics_df["class_val"] == URBAN_CLASS_VAL]
    .apply(np.log10)
    .hist(column="area", label="urban", density=True, alpha=0.66)
)
patch_metrics_df[patch_metrics_df["class_val"] == AGRICULTURAL_CLASS_VAL].apply(
    np.log10
).hist(column="area", ax=ax, label="agricultural", density=True, alpha=0.66)

ax.item().get_xaxis().set_major_formatter(
    mpl.ticker.FuncFormatter(lambda x, p: "10^%d" % x)
)
ax.item().legend()

In [ ]:
ax = (
    patch_metrics_df[patch_metrics_df["class_val"] == 1]
    .apply(np.log10)
    .hist(column="area", label="urban", density=True)
)

(class-metrics-df)=
### Class-level metrics

The metrics can also be computed at the class level, that is, aggregating over all patches of a land use/cover class

In [ ]:
class_metrics_df = ls.compute_class_metrics_df()
class_metrics_df

### Landscape-level metrics

Finally, the metrics can also be computed at the landscape level, that is, aggregating over all patches of the landscape

In [ ]:
landscape_metrics_df = ls.compute_landscape_metrics_df()
landscape_metrics_df

## Customizing the metrics DataFrames

(subset-class-metrics-df)=
### Selecting the metrics to compute

Some metrics can be expensive to compute. If you are only interested in computing a subset of the metrics implemented within pylandstats, you can specify it in each respective method, that is, `Landscape.compute_patch_metrics_df`, `Landscape.compute_patch_metrics_df` and/or `Landscape.compute_class_metrics_df` through the `metrics` argument (see [the documentation on "Computing metrics data frames"](https://pylandstats.readthedocs.io/en/latest/landscape.html#computing-metrics-data-frames). For instance:

In [ ]:
subset_class_metrics_df = ls.compute_class_metrics_df(
    metrics=["proportion_of_landscape", "edge_density"]
)
subset_class_metrics_df

### Customizing how each metric is computed

The default arguments correspond to how the metrics are defined within FRAGSTATS. Nevertheless, some metrics allow some variations in their definition. For instance, the `edge_density` above allows us to choose whether we consider the landscape boundary to be an edge (by default, as in FRAGSTATS, we do not, since we only consider edges between land use/cover classes), or whether we want the area to be converted to hectares (by default, as in FRAGSTATS, we do).

In [ ]:
print(
    "Edge density (without boundary, meters of edge per hectare):\n{}\n".format(
        ls.edge_density()
    )
)

print(
    "Edge density (with boundary, meters of edge per hectare):\n{}\n".format(
        ls.edge_density(count_boundary=True)
    )
)

print(
    "Edge density (with boundary, meters of edge per square meter):\n{}".format(
        ls.edge_density(count_boundary=True, hectares=False)
    )
)

For more details, see the documentation for each metric's method.

If we want to obtain a patch, class or landscape-level DataFrame with some customized metrics, instead of manually calling each metric's method with its respective parameters, we can use the `metrics_kwargs` argument of the `Landscape.compute_patch_metrics_df`, `Landscape.compute_patch_metrics_df` and/or `Landscape.compute_class_metrics_df` to set the keyword arguments to be passed to the some metric methods. For instance, if we wanted `proportion_of_landscape` as a fraction instead of a percentage and `edge_density` to include the boundary, we can do it as follows:

In [ ]:
custom_class_metrics_df = ls.compute_class_metrics_df(
    metrics_kwargs={
        "proportion_of_landscape": {"percent": False},
        "edge_density": {"count_boundary": True},
    }
)
custom_class_metrics_df

Note that the values for `proportion_of_landscape` and `edge_density` are different now than when we computed them with the default arguments for their respective methods (in the [class_metrics_df](#class-metrics-df) and [subset_class_metrics_df](#subset-class-metrics-df) above).

Note also that the `custom_class_metrics_df` does not only feature `proportion_of_landscape` and `edge_density` but features all the available metrics instead. This is because the `metrics_kwargs` argument does not imply that only the the metrics defined on it will be computed, only that the metrics defined on it will be computed with the specified arguments.

We might choose to only compute a reduced set of metrics, some of which with non-default arguments, by setting both the `metrics` and `metric_kwargs` arguments:

In [ ]:
custom_subset_class_metrics_df = ls.compute_class_metrics_df(
    metrics=["proportion_of_landscape", "edge_density", "fractal_dimension_am"],
    metrics_kwargs={
        "proportion_of_landscape": {"percent": False},
        "edge_density": {"count_boundary": True},
    },
)
custom_subset_class_metrics_df

The same could be done for the `Landscape.compute_patch_metrics_df` or `Landscape.compute_landscape_metrics_df` methods. Check the documentation of each metric's method for more details on how they might be customized through their arguments.